# 06 — Parcel Zonal Stats

**Learning goal:** Bridge the raster world to tabular. Compute per-parcel statistics
from every processed raster, write to Parquet, and verify DuckDB can consume it.

This is a dry run of the production pipeline. Every downstream model (LSTM, XGBoost)
consumes the Parquet file this notebook produces.

In [ ]:
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
import rasterio

try:
    from rasterstats import zonal_stats
    HAS_RASTERSTATS = True
except ImportError:
    HAS_RASTERSTATS = False
    print("rasterstats not installed: pip install rasterstats")

try:
    import duckdb
    HAS_DUCKDB = True
except ImportError:
    HAS_DUCKDB = False
    print("duckdb not installed: pip install duckdb")

# Project paths
DATA_DIR = Path("../data")
PROCESSED_DIR = DATA_DIR / "processed"
RESULTS_DIR = DATA_DIR / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_PARQUET = RESULTS_DIR / "parcel_zonal_stats.parquet"

AOI_BBOX = [-105.23, 39.915, -105.12, 39.98]

print(f"PROCESSED_DIR: {PROCESSED_DIR.resolve()}")
print(f"Output Parquet: {OUTPUT_PARQUET.resolve()}")
print(f"rasterstats: {HAS_RASTERSTATS}, duckdb: {HAS_DUCKDB}")

## Compute Zonal Stats for All Processed Rasters

We loop over every `.tif` file in `data/processed/`, organized by subdirectory (sar, landsat, etc.)
and observation date. For each raster, we compute per-parcel:
- **mean** — central tendency of the signal
- **std** — within-parcel variability (mixed pixels)
- **count** — number of valid pixels (quality flag)

In [ ]:
# Load parcel geometries
parcel_paths = [
    DATA_DIR / "raw" / "parcels.geojson",
    DATA_DIR / "raw" / "boulder_parcels.geojson",
    DATA_DIR / "tabular" / "parcels.geojson",
]

parcels = None
for p in parcel_paths:
    if p.exists():
        parcels = gpd.read_file(p)
        print(f"Loaded {len(parcels)} parcels from {p}")
        break

if parcels is None:
    print("No parcel file found. Creating synthetic parcels for demonstration.")
    print("In production, download Boulder County parcel shapefile.")

    # Create a grid of synthetic parcels over the AOI for demo purposes
    from shapely.geometry import box
    xmin, ymin, xmax, ymax = AOI_BBOX
    cell_size = 0.001  # ~100m parcels
    rows_list = []
    pid = 0
    y = ymin
    while y < ymax:
        x = xmin
        while x < xmax:
            rows_list.append({
                "parcel_id": f"SYNTH_{pid:05d}",
                "geometry": box(x, y, x + cell_size, y + cell_size),
            })
            pid += 1
            x += cell_size
        y += cell_size
    parcels = gpd.GeoDataFrame(rows_list, crs="EPSG:4326")
    print(f"Created {len(parcels)} synthetic parcels")

# Ensure CRS is EPSG:4326 for consistency
parcels = parcels.to_crs("EPSG:4326")

# Identify parcel ID column
parcel_id_col = None
for col in ["parcel_id", "PARCEL_ID", "parcelid", "APN", "PIN"]:
    if col in parcels.columns:
        parcel_id_col = col
        break
if parcel_id_col is None:
    parcel_id_col = "parcel_id"
    parcels["parcel_id"] = [f"P{i:05d}" for i in range(len(parcels))]

print(f"Parcel ID column: {parcel_id_col}")
print(f"Parcels shape: {parcels.shape}")
parcels.head()

In [ ]:
import re


def discover_rasters(base_dir):
    """Find all .tif files in processed directory and parse metadata from paths.
    Expected structure: data/processed/{raster_type}/{observation_date}_{name}.tif
    or data/processed/{raster_type}/{name}.tif
    """
    rasters = []
    base = Path(base_dir)

    if not base.exists():
        print(f"Warning: {base} does not exist")
        return rasters

    for tif_path in sorted(base.rglob("*.tif")):
        # Parse raster type from parent directory
        rel = tif_path.relative_to(base)
        parts = rel.parts
        raster_type = parts[0] if len(parts) > 1 else "unknown"
        stem = tif_path.stem

        # Try to extract date from filename (YYYY-MM or YYYYMM patterns)
        date_match = re.search(r"(20\d{2}[-_]?\d{2})", stem)
        obs_date = date_match.group(1).replace("_", "-") if date_match else "unknown"

        rasters.append({
            "path": tif_path,
            "raster_type": raster_type,
            "observation_date": obs_date,
            "name": stem,
        })

    return rasters


rasters = discover_rasters(PROCESSED_DIR)
print(f"Found {len(rasters)} raster files in {PROCESSED_DIR}")
for r in rasters:
    print(f"  {r['raster_type']:12s} | {r['observation_date']:10s} | {r['name']}")

if not rasters:
    print("\nNo rasters found in data/processed/.")
    print("Run notebooks 03-05 first to generate processed rasters.")
    print("\nCreating a demo raster for pipeline testing...")

    # Create a minimal demo raster so the pipeline can be tested end-to-end
    from rasterio.transform import from_bounds
    demo_dir = PROCESSED_DIR / "sar"
    demo_dir.mkdir(parents=True, exist_ok=True)
    demo_path = demo_dir / "2022-01_vv_change_db.tif"

    h, w = 100, 100
    transform = from_bounds(*AOI_BBOX, w, h)
    demo_data = np.random.randn(h, w).astype(np.float32) * 3  # Fake VV change

    with rasterio.open(
        demo_path, "w", driver="GTiff",
        height=h, width=w, count=1, dtype="float32",
        crs="EPSG:4326", transform=transform,
    ) as dst:
        dst.write(demo_data, 1)

    print(f"Created demo raster: {demo_path}")
    rasters = discover_rasters(PROCESSED_DIR)
    print(f"Now have {len(rasters)} raster files")

In [ ]:
# Compute zonal stats for all rasters
all_rows = []

for raster_info in rasters:
    tif_path = raster_info["path"]
    raster_type = raster_info["raster_type"]
    obs_date = raster_info["observation_date"]
    raster_name = raster_info["name"]

    print(f"Processing: {raster_name} ...", end=" ")

    try:
        with rasterio.open(tif_path) as src:
            raster_data = src.read(1)
            raster_transform = src.transform
            raster_crs = src.crs

        # Reproject parcels to raster CRS if needed
        parcels_proj = parcels.to_crs(raster_crs)

        stats = zonal_stats(
            parcels_proj,
            raster_data,
            affine=raster_transform,
            stats=["mean", "std", "count"],
            nodata=np.nan,
        )

        for i, stat in enumerate(stats):
            all_rows.append({
                "parcel_id": parcels.iloc[i][parcel_id_col],
                "raster": raster_name,
                "raster_type": raster_type,
                "observation_date": obs_date,
                "mean": stat.get("mean"),
                "std": stat.get("std"),
                "pixel_count": stat.get("count"),
            })

        print(f"OK ({len(stats)} parcels)")

    except Exception as e:
        print(f"FAILED: {e}")

print(f"\nTotal rows collected: {len(all_rows)}")

## Write to Parquet — This Is What dbt Consumes

The Parquet file is the handoff between the raster/geospatial world and the
tabular/analytics world. Every downstream pipeline reads from this file.

In [ ]:
df = pd.DataFrame(all_rows)
print(f"DataFrame shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"\nRaster types: {df['raster_type'].unique().tolist()}")
print(f"Observation dates: {df['observation_date'].unique().tolist()}")
print(f"Unique parcels: {df['parcel_id'].nunique()}")
print(f"\nNull mean values: {df['mean'].isna().sum()} / {len(df)}")

# Write to Parquet
df.to_parquet(OUTPUT_PARQUET, index=False, engine="pyarrow")

file_size_mb = OUTPUT_PARQUET.stat().st_size / (1024 * 1024)
print(f"\nWritten to: {OUTPUT_PARQUET}")
print(f"File size: {file_size_mb:.2f} MB")
print("\nSample rows:")
df.head(10)

## Verify DuckDB Can Read and Aggregate

DuckDB reads Parquet natively — no loading step needed.
This verifies the file is well-formed and shows summary statistics.

In [ ]:
if HAS_DUCKDB:
    con = duckdb.connect()

    # Basic read test
    result = con.execute(f"""
        SELECT count(*) as total_rows,
               count(DISTINCT parcel_id) as n_parcels,
               count(DISTINCT raster) as n_rasters
        FROM '{OUTPUT_PARQUET}'
    """).fetchdf()
    print("=== File Summary ===")
    print(result.to_string(index=False))

    # Aggregation by raster type and observation date
    result2 = con.execute(f"""
        SELECT raster_type,
               observation_date,
               count(*) as n_rows,
               round(avg(mean), 4) as avg_mean,
               round(avg(std), 4) as avg_std,
               round(avg(pixel_count), 1) as avg_pixels
        FROM '{OUTPUT_PARQUET}'
        GROUP BY raster_type, observation_date
        ORDER BY raster_type, observation_date
    """).fetchdf()
    print("\n=== Aggregation by Raster Type x Date ===")
    print(result2.to_string(index=False))

    # Check for parcels with mostly null values (data quality)
    result3 = con.execute(f"""
        SELECT parcel_id,
               count(*) as n_observations,
               sum(CASE WHEN mean IS NULL THEN 1 ELSE 0 END) as n_nulls
        FROM '{OUTPUT_PARQUET}'
        GROUP BY parcel_id
        HAVING n_nulls > 0
        ORDER BY n_nulls DESC
        LIMIT 10
    """).fetchdf()
    print("\n=== Parcels with Null Values (top 10) ===")
    if len(result3) > 0:
        print(result3.to_string(index=False))
    else:
        print("No parcels with null values — clean data.")

    con.close()
    print("\nDuckDB verification complete.")
else:
    print("DuckDB not installed. Install with: pip install duckdb")
    print("Falling back to pandas verification:")
    verify_df = pd.read_parquet(OUTPUT_PARQUET)
    print(f"Read back {len(verify_df)} rows")
    print(verify_df.groupby(["raster_type", "observation_date"]).agg(
        n_rows=("mean", "count"),
        avg_mean=("mean", "mean"),
        avg_std=("std", "mean"),
    ).round(4))